# 5b. Visualising the source estimates using pyvista (3D brains)

In this notebook we look at the computed source estimates on the cortical surface in 3D. 

<div class="alert alert-success">
    <b>Learning Objectives</b>:
     <ul>
      <li>Visualizing the computed reconstructed activity.</li>
      <li>Interpreting our results: do we see a meaningful contrast for our condition perceived vs. unperceived?</li>
      <li>Navigate through the 3D plotter. </li>
    </ul>
</div>

The imports and settings are slightly more complex in this notebook, since 3D rendering in JupyterHub requires special settings. 

In [ ]:
# imports
import os
import sys
os.environ["PYVISTA_OFF_SCREEN"] = "true"
# on a Linux server without a display, render with OSMesa (system package libosmesa6)
if sys.platform.startswith("linux") and not os.environ.get("DISPLAY"):
    os.environ["VTK_DEFAULT_OPENGL_WINDOW"] = "vtkOSOpenGLRenderWindow"

import numpy as np
import matplotlib.pyplot as plt

import pyvista as pv
pv.OFF_SCREEN = True

import mne
# render the 3D brains inline in JupyterLab
mne.viz.set_3d_backend("notebook")

# start the trame server that streams the 3D views to the browser
from pyvista.trame.jupyter import launch_server
await launch_server().ready

# the time viewer widgets change the scene in Python, push each change to the view in the browser
from mne.viz.backends._notebook import _Renderer
from pyvista.trame.ui import _VIEWERS

if not hasattr(_Renderer, "_update_in_kernel"):
    _Renderer._update_in_kernel = _Renderer._update

    def _update_and_push(self):
        self._update_in_kernel()
        viewer = _VIEWERS.get(self.plotter._id_name)
        if viewer is not None:
            viewer.update()

    _Renderer._update = _update_and_push


# data paths 
subID = 24
data_path = "data"
subject_path = os.path.join(f"sub-0{subID}", "ses-mecha", "eeg")
base = os.path.join("..", data_path, subject_path, f"sub-0{subID}_ses-mecha_task-NT_")

# fsaverage: downloads it on first use, otherwise just returns the path
fs_dir = mne.datasets.fetch_fsaverage()
subjects_dir = os.path.dirname(fs_dir)
print(subjects_dir)

## 5.1 Morphing to fsaverage

We morph the source estimate onto *fsaverage*, which we downloaded previously, and with the precomputed morph.

In [ ]:
# load the source estimate and the precomputed morph
stc = mne.read_source_estimate(base + "MNE-all")
morph = mne.read_source_morph(base + "morph.h5")

# apply the morph: individual source space -> fsaverage
stc_fs = morph.apply(stc)
print(stc_fs)

## 5.2. Plotting the contrast on the brain

The perceived-minus-unperceived contrast, morphed to fsaverage, plotted on the brain.

`stc.plot()` paints the estimate on the cortical surface. The most important arguments:

- `surface="inflated"` - unfolds the cortex so that activity inside the sulci stays visible.
- `hemi="both"` - both hemispheres (`"lh"`, `"rh"` or `"split"` also work).
- `initial_time` - which time point to show when the figure opens.
- `time_viewer=True` - adds a time slider, so you can step through the epoch or play it as
  a movie.

Remember from notebook 04 that our solution is fixed-orientation with respect to cortical sourface, so the values are
**signed**: red and blue mean current flowing out of and into the cortical surface, not
"more" and "less" activity.

In [ ]:
stc_difference = mne.read_source_estimate(base + "MNE-difference")
stc_difference_fs = morph.apply(stc_difference)

diff_vertex, diff_time = stc_difference_fs.get_peak(tmin=0.05, tmax=0.3)
print(f"largest difference at {diff_time * 1000:.0f} ms")

brain = stc_difference_fs.plot(
    subject="fsaverage",
    subjects_dir=subjects_dir,
    surface="inflated",
    hemi="both",
    initial_time=diff_time,
    clim=dict(kind="percent", pos_lims=[97, 99, 99.95]),
    time_viewer=True,
)

<div class="alert alert-warning">
    <b>Exercise</b>:
    <ul>
        <li>In which brain areas do you see activation?</li>
        <li>What does red and blue activation mean?</li>
    </ul>
</div>

### 5.3 The Colour scale

The color scale of the plots determine how many sources are shown and how dark they are plotted.
`clim` sets it: `kind="percent"` puts the three thresholds (transparent / saturating / maximum) at
percentiles of the data, so raising them shows fewer, more focal sources. Because our
estimates are signed we pass `pos_lims`, which mirrors the same limits to the negative side.
`kind="value"` takes absolute numbers instead - that is what you need when several
conditions have to share one scale.

Example: with `pos_lims=[95, 99, 99.95]`, values below the 95th percentile become transparent, and the colour scale saturates at the 99.95th percentile.

<div class="alert alert-warning">
    <b>Exercise</b>:
    Experiment with different values of the "pos_lims" parameter. What do you observe?
</div>

In [ ]:
brain = stc_difference_fs.plot(
    subject="fsaverage",
    subjects_dir=subjects_dir,
    surface="inflated",
    hemi="both",
    initial_time=diff_time,
    clim=dict(kind="percent", pos_lims=[95, 99, 99.95]),
    time_viewer=True,
)

## 5.4 Time course of a single source

We have a look at a single vertex activity. Play around with the vertex number to show different reconstructed traces.

In [ ]:
peak_idx, peak_time = stc_difference_fs.get_peak(tmin=0.05, tmax=0.3, vert_as_index=True)
n_lh = len(stc_fs.vertices[0])
hemi = "lh" if peak_idx < n_lh else "rh"
print(f"strongest source: row {peak_idx} ({hemi}), peaking at {peak_time * 1000:.0f} ms")

plt.plot(stc_difference_fs.times[100:], stc_difference_fs.data[peak_idx][100:])
plt.axvline(0, color="k", linestyle="--")
plt.axvline(peak_time, color="r", linestyle=":")
plt.xlabel("time (s)")
plt.ylabel("amplitude (Am)")
plt.title(f"peak source ({hemi})")
plt.show()

## 5.5 Anatomical parcels

Instead of single vertices we can average within anatomical regions. `aparc` is the
Desikan-Killiany parcellation that comes with FreeSurfer: 68 cortical labels, defined on
fsaverage and therefore directly usable on our morphed estimate.

Here, we want to compute the activity of the primary somatosensory cortex.
How to average the vertices can be chosen via the argument `mode`.
For example: `mode="mean_flip"`, which flips the sign of vertices whose surface normal points the other way
before averaging. 

Remember: There is still source leakage, so a label time course is **not** only "the activity
of that region". 

<div class="alert alert-warning">
    <b>Exercise</b>:
     <ul>
      <li>In which cases is it reasonable to choose `mean_flip` or `pca_flip` over `mean` as averaging option?</li>
      <li>Which option would you chose for the primary somatosensory cortex? (Tipp: look up the geometry of S1 and the pattern of the source reconstructed activity)</li>
    </ul>
</div>

In [ ]:
# the parcellation and the source space it is defined on
labels = mne.read_labels_from_annot("fsaverage", parc="aparc", subjects_dir=subjects_dir)
stc_difference_fs = mne.read_source_spaces(
    os.path.join(subjects_dir, "fsaverage", "bem", "fsaverage-ico-5-src.fif")
)

# a few regions that are plausible for a tactile detection task
roi_names = ["postcentral-lh", "postcentral-rh"]
rois = [label for label in labels if label.name in roi_names]

label_tc = mne.extract_label_time_course(stc_difference_fs, rois, stc_difference_fs, mode="pca_flip")
print("label time courses:", label_tc.shape)

In [ ]:
for name, time_course in zip([label.name for label in rois], label_tc):
    plt.plot(stc_difference_fs.times, time_course, label=name)
plt.axvline(0, color="k", linestyle="--")
plt.xlabel("time (s)")
plt.ylabel("amplitude (Am)")
plt.legend()
plt.title("label time courses")

In [ ]:
# visualize where these labels sit on the brain
brain = stc_difference_fs.plot(
    subject="fsaverage",
    subjects_dir=subjects_dir,
    surface="inflated",
    hemi="both",
    initial_time=peak_time,
    time_viewer=False,
)
for label in rois:
    brain.add_label(label, borders=True)

# precuneus sits on the medial wall, so it does not show up in a lateral view
print("ROIs:")
for label in rois:
    print(f"  {label.name}")

## 5.6 Comparing LCMV and MNE inverse solution

In [ ]:
# load the LCMV contrast (perceived - unperceived) from the beamformer solution notebook
stc_lcmv_difference = mne.read_source_estimate(base + "lcmv-difference")
stc_lcmv_difference_fs = morph.apply(stc_lcmv_difference)

lcmv_vertex, lcmv_time = stc_lcmv_difference_fs.get_peak(tmin=0.05, tmax=0.3)
print(f"largest LCMV difference at {lcmv_time * 1000:.0f} ms (MNE: {diff_time * 1000:.0f} ms)")

# plot at the same time point as the MNE contrast above, so the two maps can be compared.
# LCMV (unit-noise-gain) and MNE values have different units, so each gets its own colour scale.
brain = stc_lcmv_difference_fs.plot(
    subject="fsaverage",
    subjects_dir=subjects_dir,
    surface="inflated",
    hemi="both",
    initial_time=diff_time,
    clim=dict(kind="percent", pos_lims=[95, 99, 99.95]),
    time_viewer=True,
)

<div class="alert alert-warning">
    <b>Exercise</b>:
    <ul>
        <li>What differences do you spot when comparing the solutions from the LCMV and MNE beamformer?</li>
        <li>What might be the reasons for different outcomes?</li>
    </ul>
</div>

## 5.7 Source activation at the N140 component
According to previous literature, the 140 component reflects conscious perception of somatosensory stimuli. Now, let's plot the activated sources at that timepoint for both of our methods.

In [ ]:
n140_time = 0.14  # s

# one plot per inverse method; each gets its own colour scale, since MNE is in Am and LCMV in pseudo-Z
for name, stc_contrast in [("MNE", stc_difference_fs), ("LCMV", stc_lcmv_difference_fs)]:
    brain = stc_contrast.plot(
        subject="fsaverage",
        subjects_dir=subjects_dir,
        surface="inflated",
        hemi="both",
        initial_time=n140_time,
        clim=dict(kind="percent", pos_lims=[95, 99, 99.95]),
        time_viewer=True,
        title=name,
    )